# W01 — What is a Qubit?
**Q-BITS · BITS Pilani Dubai Campus · Track 0: Foundations**

---

This notebook introduces the qubit — the fundamental unit of quantum information. We interleave theory with code throughout. No prior quantum knowledge required.

**By the end of this notebook you will be able to:**
- Describe a qubit state mathematically
- Visualise states on the Bloch sphere
- Understand what measurement does
- Run a simple single-qubit circuit in Qiskit

In [ ]:
# Install dependencies if needed
# !pip install qiskit qiskit-aer matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer import AerSimulator

print('Imports successful.')

---

## 1. Classical bits vs qubits

A classical bit is always either 0 or 1. That's it — two states, one value at any given time.

A **qubit** is different. Before measurement, it can exist in a **superposition** of |0⟩ and |1⟩ simultaneously. The general state of a qubit is:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

where $\alpha$ and $\beta$ are complex numbers called **amplitudes**, subject to the constraint:

$$|\alpha|^2 + |\beta|^2 = 1$$

This is the **normalisation condition** — it ensures probabilities sum to 1.

### What superposition is NOT
Superposition does not mean the qubit is "both 0 and 1 at once" in a classical sense. It means the qubit is in a quantum state that, upon measurement, yields 0 with probability $|\alpha|^2$ and 1 with probability $|\beta|^2$.

In [ ]:
# Represent qubit states as numpy vectors
# |0> and |1> are the computational basis states

ket_0 = np.array([1, 0])  # |0>
ket_1 = np.array([0, 1])  # |1>

# A superposition state: equal probability of 0 and 1
# alpha = beta = 1/sqrt(2)
alpha = 1 / np.sqrt(2)
beta  = 1 / np.sqrt(2)
psi   = alpha * ket_0 + beta * ket_1

print(f'|0>  = {ket_0}')
print(f'|1>  = {ket_1}')
print(f'|ψ>  = {psi}')
print(f'Normalisation check: |α|² + |β|² = {abs(alpha)**2 + abs(beta)**2:.4f}')

---

## 2. Dirac Notation

Quantum mechanics uses **bra-ket (Dirac) notation**:

| Symbol | Name | Meaning |
|---|---|---|
| $|\psi\rangle$ | ket | a quantum state (column vector) |
| $\langle\psi|$ | bra | conjugate transpose of a ket (row vector) |
| $\langle\phi|\psi\rangle$ | bracket | inner product of two states |

The two computational basis states are:
$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \quad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$

You will see this notation throughout every workshop in this series.

In [ ]:
# Inner product: <0|0> should be 1, <0|1> should be 0 (orthonormal basis)

inner_00 = np.dot(ket_0.conj(), ket_0)
inner_01 = np.dot(ket_0.conj(), ket_1)
inner_11 = np.dot(ket_1.conj(), ket_1)

print(f'<0|0> = {inner_00}')  # 1
print(f'<0|1> = {inner_01}')  # 0  — orthogonal
print(f'<1|1> = {inner_11}')  # 1

---

## 3. The Bloch Sphere

Any single-qubit state can be visualised as a point on the surface of a unit sphere called the **Bloch sphere**. Using the normalisation constraint and the fact that global phase is unobservable, we can write any qubit state as:

$$|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle$$

where:
- $\theta \in [0, \pi]$ is the polar angle (latitude)
- $\phi \in [0, 2\pi)$ is the azimuthal angle (longitude)

Key points on the Bloch sphere:

| State | Position |
|---|---|
| $|0\rangle$ | North pole |
| $|1\rangle$ | South pole |
| $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$ | +X axis |
| $|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle-|1\rangle)$ | −X axis |
| $|i\rangle = \frac{1}{\sqrt{2}}(|0\rangle+i|1\rangle)$ | +Y axis |

In [ ]:
# Visualise |0>, |1>, and |+> on the Bloch sphere using Qiskit

sv_0 = Statevector([1, 0])                          # |0>
sv_1 = Statevector([0, 1])                          # |1>
sv_plus = Statevector([1/np.sqrt(2), 1/np.sqrt(2)]) # |+>

fig, axes = plt.subplots(1, 3, figsize=(12, 4),
                          subplot_kw={'projection': '3d'})

for ax, sv, label in zip(axes,
                         [sv_0, sv_1, sv_plus],
                         ['|0⟩', '|1⟩', '|+⟩']):
    plot_bloch_multivector(sv, ax=ax)
    ax.set_title(label, fontsize=14)

plt.tight_layout()
plt.show()

---

## 4. Measurement and the Born Rule

Measuring a qubit **collapses** it to either |0⟩ or |1⟩. This is irreversible.

The **Born rule** gives the probabilities:

$$P(0) = |\alpha|^2, \qquad P(1) = |\beta|^2$$

For the state $|+\rangle = \frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle$:

$$P(0) = \left|\frac{1}{\sqrt{2}}\right|^2 = \frac{1}{2}, \qquad P(1) = \frac{1}{2}$$

After measurement, the qubit is no longer in superposition — it is now definitively 0 or 1. **You cannot un-measure a qubit.**

In [ ]:
# Build a circuit that prepares |+> and measures it
# H gate puts |0> into equal superposition (we cover gates in W02)

qc = QuantumCircuit(1, 1)
qc.h(0)        # apply Hadamard — creates |+>
qc.measure(0, 0)

print(qc.draw('text'))

In [ ]:
# Run on the Aer simulator — 1024 shots

simulator = AerSimulator()
job = simulator.run(qc, shots=1024)
counts = job.result().get_counts()

print('Measurement results:', counts)
plot_histogram(counts)

You should see approximately 50% `0` and 50% `1` — matching our Born rule prediction. The exact split varies each run because quantum measurement is fundamentally probabilistic.

---

## 5. Exercises

Work through these before W02.

**Exercise 1.** Write the state $|\psi\rangle = \frac{1}{2}|0\rangle + \frac{\sqrt{3}}{2}|1\rangle$ as a numpy vector. Verify it is normalised.

**Exercise 2.** What are $P(0)$ and $P(1)$ for the state above? Confirm they sum to 1.

**Exercise 3.** Modify the circuit above to measure |0⟩ directly (skip the H gate). What do you expect? Run it and verify.

**Exercise 4.** Look up the state $|i\rangle = \frac{1}{\sqrt{2}}(|0\rangle + i|1\rangle)$. Represent it as a numpy vector and plot it on the Bloch sphere. Where does it sit?

In [ ]:
# Your answers here


---

## Summary

| Concept | Key point |
|---|---|
| Qubit state | $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$, with $|\alpha|^2 + |\beta|^2 = 1$ |
| Superposition | A qubit can be in a combination of |0⟩ and |1⟩ before measurement |
| Bloch sphere | Every single-qubit state maps to a point on the unit sphere |
| Measurement | Collapses the state; yields 0 with prob $|\alpha|^2$, 1 with prob $|\beta|^2$ |
| Born rule | $P(\text{outcome}) = |\text{amplitude}|^2$ |

**Next workshop:** [W02 — Quantum Gates 101](../W02/notebook.ipynb)

---
*Q-BITS · BITS Pilani Dubai Campus*